# TESSERA-IDR

**T**iling-based **E**xploration of **S**equence **S**ignatures for **E**xtended **R**esidue **A**nnotation in Intrinsically Disordered Regions

Developed by Alexandre P. Magalhaes. Contact: [magalhae@molgen.mpg.de](mailto:magalhae@molgen.mpg.de)

Assign intrinsically disordered regions (IDRs) to the 30 published GIN centroids
(grammars of IDRs) with the NARDINI+ feature pipeline, then look at the result
residue by residue and in the context of the whole protein.

| Mode | What it answers | Code |
|---|---|---|
| **Multi-block** | Does the IDR contain several grammars, and where does each one sit? A 70 aa window slides along the IDR (step 10), every window is assigned, and blocks grow from confident seed windows. | `src/nardini_MultiBlock_v099.py` |
| **Full sequence** | Which single grammar fits the IDR as a whole? | `src/nardini_fullSeq_numba_v096.py` |

Both use the corrected ω/κ window kernel
([issue #2](https://github.com/kierstenruff/RUFF_KING_Grammars_of_IDRs_using_NARDINI-/issues/2))
and the original centroids, unchanged. The multi-block defaults are the settings
of the published human run, and for human IDRs you can also show the published
blocks themselves (`results/multiblock/block_summary_seedExpantion.20260909.csv.gz`)
without recomputing anything.

**Human IDRs only.** The 30 GIN centroids, the compositional z-score
normalisation and every reference table here come from the human IDRome. Use
this notebook for human proteins: a sequence from another species is still
scored, but against human statistics, so its GIN assignments and rankings are
not meaningful. Support for other species will be added soon.

**How to use it.** Run the cells from top to bottom (*Runtime → Run all* works
once the inputs in step 2 are set). Each step has a form on the right; the code
is hidden but one click away.

**Cite.** If you use these results, please cite the methods they rest on:

- NARDINI: M.C. Cohan, M.K. Shinn, J.M. Lalmansingh, R.V. Pappu (2022). Uncovering non-random binary patterns within sequences of intrinsically disordered proteins. *J. Mol. Biol.* 434, 167373.
- NARDINI+: M.R. King, K.M. Ruff, *et al.*, R.V. Pappu (2024). Macromolecular condensation organizes nucleolar sub-phases to set up a pH gradient. *Cell* 187, 1889–1906.
- GIN resource and centroids: K.M. Ruff, M.R. King, *et al.*, R.V. Pappu (2025). Molecular grammars of predicted intrinsically disordered regions that span the human proteome. *Cell*.
- localCIDER, used for the compositional features: A.S. Holehouse, R.K. Das, J.N. Ahad, M.O.G. Richardson, R.V. Pappu (2017). CIDER: resources to analyze sequence-ensemble relationships of intrinsically disordered proteins. *Biophys. J.* 112, 16–21.

## 1 · Set up the runtime

The notebook needs the repository next to it. By default it downloads it from
GitHub. You can also point to a folder in your Google Drive, or clear both
fields to upload a `.zip` of the repository folder. Installing Arial takes about 30 s and makes every
figure use it; PDFs keep their text editable either way.

In [ ]:
#@title Set up { display-mode: "form" }
REPO_URL = "https://github.com/alexpmagalhaes/TESSERA-IDR"  #@param {type:"string"}
REPO_ON_DRIVE = ""  #@param {type:"string"}
INSTALL_ARIAL = True  #@param {type:"boolean"}
FIGURE_FORMATS = "png, pdf"  #@param ["png, pdf", "png, pdf, svg", "svg"]

import os, sys, glob, shutil, subprocess, zipfile
from pathlib import Path

REPO_DIR = Path("/content/TESSERA-IDR")
if "/YOUR-" in REPO_URL:
    REPO_URL = ""  # Repository address not filled in yet: fall back to the zip upload.
IN_COLAB = "google.colab" in sys.modules


def find_repo(root):
    hits = glob.glob(f"{root}/**/colab/nardini_colab.py", recursive=True)
    return Path(hits[0]).parent.parent if hits else None


if not (REPO_DIR / "colab" / "nardini_colab.py").exists():
    if REPO_URL.strip():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL.strip(), str(REPO_DIR)], check=True)
    elif REPO_ON_DRIVE.strip():
        from google.colab import drive
        drive.mount("/content/drive")
        source = Path(REPO_ON_DRIVE.strip())
        if not source.is_absolute():
            source = Path("/content/drive/MyDrive") / source
        shutil.copytree(source, REPO_DIR, ignore=shutil.ignore_patterns("*.out_windows.csv*", ".git"))
    else:
        from google.colab import files
        print("Choose the repository .zip")
        uploaded = files.upload()
        archive = next(iter(uploaded))
        with zipfile.ZipFile(archive) as zf:
            zf.extractall("/content/_repo_upload")
        found = find_repo("/content/_repo_upload")
        if found is None:
            raise FileNotFoundError("The zip has no colab/nardini_colab.py; is it the TESSERA-IDR repository?")
        shutil.move(str(found), REPO_DIR)

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "localcider", "multiprocess"], check=True)
    if INSTALL_ARIAL and not glob.glob("/usr/share/fonts/truetype/msttcorefonts/[Aa]rial.ttf"):
        subprocess.run("echo ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true"
                       " | debconf-set-selections && apt-get -qq install -y ttf-mscorefonts-installer > /dev/null",
                       shell=True)

sys.path.insert(0, str(REPO_DIR / "colab"))
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
import nardini_colab as nc

FONT = nc.setup_fonts()
FORMATS = tuple(f.strip() for f in FIGURE_FORMATS.split(","))
print(f"Repository: {REPO_DIR}")
print(f"CPUs: {os.cpu_count()}   Figure font: {FONT}")

## 2 · Sequences

Pick a source:

- **Example** – 7 human IDRs, about 15 s for both modes.
- **Human IDRs by UniProt ID or gene name** – every IDR ≥ 100 aa of the listed proteins, from the bundled human set (gene names are matched regardless of case). Domains and protein lengths come with them.
- **Paste FASTA** – edit the text in the next cell. Optional header fields place the IDR in its protein: `>MyIDR uniprot=Q9Y388 start=112 gene=RBMX2`. UniProt headers (`>sp|Q9Y388|RBMX2_HUMAN`) are read too.
- **Upload file** – FASTA, or a CSV/TSV with `IDR_ID` and `Sequence` (or `IDR_Sequence`) columns; extra columns such as `UniprotID`, `Gene name`, `IDR_START` are carried through to every output.

Pasted or uploaded sequences should be human (see the note at the top). Only the
20 standard amino acids are scored. The multi-block mode needs at least
50 residues; blocks shorter than the block-length filter (70 aa by default) are dropped.

In [ ]:
#@title Paste FASTA here (used only when the source is "Paste FASTA") { display-mode: "form" }
FASTA = """
>Q9Y388_IDR_1 uniprot=Q9Y388 start=112 gene=RBMX2
SNYRAPKDSEEIDDVTRQLQEKGCGARTPSPSLSESSEDEKPTKKHKKDKKEKKKKKKEKEKADREVQAEQPSSSSPRRKTV
KEKDDTGPKKHSSKNSERAQKSEPREGQKLPKSRTAYSGGAEDLERELKKEKPKHEHKSSSRREAREEKTRIRDRGRSSDA
HSSWYNGRSEGRSYRSRSRSRDKSHRHKRARRSRERESSNPSDRWRH
"""

In [ ]:
#@title Load sequences { display-mode: "form" }
SOURCE = "Example: 7 human IDRs"  #@param ["Example: 7 human IDRs", "Human IDRs by UniProt ID or gene name", "Paste FASTA", "Upload file"]
PROTEINS = "RBMX2, Q05519, TBXT"  #@param {type:"string"}
RUN_NAME = "my_run"  #@param {type:"string"}

if SOURCE.startswith("Example"):
    inputs = nc.load_example()
elif SOURCE.startswith("Human"):
    inputs = nc.load_human_idrs(PROTEINS.replace(";", ",").replace(" ", ",").split(","))
elif SOURCE == "Paste FASTA":
    inputs = nc.standardize_input(nc.parse_fasta(FASTA))
else:
    from google.colab import files
    uploaded = files.upload()
    name, content = next(iter(uploaded.items()))
    inputs = nc.load_uploaded(name, content)

OUT = Path("/content/nardini_results") / (RUN_NAME.strip() or "run")
INPUT_CSV = nc.write_pipeline_input(inputs, OUT / f"{OUT.name}.csv")

n_windows = int(sum(max(0, (L - 50) // 10 + 1) for L in inputs["IDR_length"]))
print(f"{len(inputs)} sequence(s), {inputs['IDR_length'].sum():,} residues, "
      f"{n_windows:,} windows for the multi-block mode")
display(inputs.drop(columns="Sequence").style.hide(axis="index"))

## 3 · Run

**Published blocks** reads the blocks of the published human run for your IDRs;
nothing is computed, so it is instant and matches the paper exactly. It works for
IDRs from the bundled human set (the example and *Human IDRs by UniProt ID*).

**Recomputing** gives the same method and settings, but patterning z-scores come
from random sequence scrambles, so a fresh run can move a block edge by a window
or flip a close call compared with the published table.

**Seed definition (multi-block).** `seed_expand` starts blocks only from windows
that sit close to their centroid (`min_dist` at or below a quantile) and far from
the runner-up (`inter_cluster_dist` at or above a quantile). With *human proteome*
as the seed reference those quantiles are taken from all 251,223 windows of the
10,371 human IDRs, so a run on a few sequences uses the same cut-offs as the
proteome-wide analysis. *This input* takes the quantiles from your own windows
instead, which is only meaningful for large inputs.

Blocks grown from seeds are kept if they are at least `MIN_BLOCK_LENGTH` residues
long, whatever their window count, as in the published run. `meanboundary` is
the alternative block builder: it keeps every window, de-noises the assignments,
drops blocks shorter than `MIN_WINDOWS` windows and splits overlaps at their midpoint.

Speed: about 10 windows per second per CPU. A free Colab runtime has 2 CPUs.

In [ ]:
#@title Run the pipelines { display-mode: "form" }
MODE = "Both"  #@param ["Both", "Multi-block (sliding window)", "Full sequence", "Published blocks only (human IDRs)"]
SHOW_PUBLISHED_BLOCKS = True  #@param {type:"boolean"}
NUM_SCRAMBLES = 100  #@param {type:"integer"}
#@markdown **Multi-block**
WINDOW_SIZE = 70  #@param {type:"integer"}
STEP_SIZE = 10  #@param {type:"integer"}
BLOCK_METHOD = "seed_expand"  #@param ["seed_expand", "meanboundary"]
SEED_REFERENCE = "Human proteome"  #@param ["Human proteome", "This input"]
SEED_MIN_DIST_QUANTILE = 0.60  #@param {type:"slider", min:0.1, max:0.9, step:0.05}
SEED_INTER_DIST_QUANTILE = 0.75  #@param {type:"slider", min:0.1, max:0.9, step:0.05}
SEED_ABSORPTION_FACTOR = 1.5  #@param {type:"number"}
MIN_BLOCK_LENGTH = 70  #@param {type:"integer"}
MIN_WINDOWS = 3  #@param {type:"integer"}
#@markdown `MIN_WINDOWS` applies to `meanboundary` only. The defaults reproduce the published run.

results = {}
if SHOW_PUBLISHED_BLOCKS or MODE.startswith("Published"):
    results["published"] = {"blocks": nc.published_blocks(inputs["IDR_ID"])}
    if not results["published"]["blocks"].empty:
        (OUT / "published").mkdir(parents=True, exist_ok=True)
        results["published"]["blocks"].to_csv(OUT / "published" / f"{OUT.name}.published_blocks.csv", index=False)

if MODE in ("Both", "Multi-block (sliding window)"):
    print("Multi-block")
    results["multiblock"] = nc.run_multiblock(
        INPUT_CSV, OUT / "multiblock",
        window_size=WINDOW_SIZE, step_size=STEP_SIZE, num_scrambles=NUM_SCRAMBLES,
        block_method=BLOCK_METHOD,
        seed_min_dist_quantile=SEED_MIN_DIST_QUANTILE,
        seed_inter_dist_quantile=SEED_INTER_DIST_QUANTILE,
        seed_absorption_factor=SEED_ABSORPTION_FACTOR,
        use_human_seed_thresholds=(SEED_REFERENCE == "Human proteome"),
        min_windows=MIN_WINDOWS, min_block_length=MIN_BLOCK_LENGTH)
    results["multiblock"]["blocks"] = nc.blocks_table(results["multiblock"]["summary"])

if MODE in ("Both", "Full sequence"):
    print("Full sequence")
    results["fullseq"] = nc.run_full_sequence(INPUT_CSV, OUT / "fullseq", num_scrambles=NUM_SCRAMBLES)
    results["fullseq"]["blocks"] = nc.blocks_table(results["fullseq"]["summary"])

clear_output()
fullseq_z = results.get("fullseq", {}).get("sequences")
idr_z = nc.idr_zscores(inputs, fullseq_z)
block_z = {}
if "published" in results:
    block_z["published"] = nc.best_windows(results["published"]["blocks"])
if "multiblock" in results:
    block_z["multiblock"] = nc.best_windows(results["multiblock"]["blocks"], results["multiblock"]["windows"])

cfg = results.get("multiblock", {}).get("cfg", {})
if cfg.get("seed_min_dist_threshold") is not None:
    print(f"Seed windows: min_dist <= {cfg['seed_min_dist_threshold']:.2f}, "
          f"inter_cluster_dist >= {cfg['seed_inter_dist_threshold']:.2f} (human proteome)")
for name, res in results.items():
    if res["blocks"].empty:
        print(f"{name}: no blocks")
        continue
    print(f"{name}: {len(res['blocks'])} block(s) in {res['blocks']['IDR_ID'].nunique() if len(res['blocks']) else 0} "
          f"of {len(inputs)} sequence(s). Files in {OUT / name}")

## 4 · Results

**Overview**: one row per IDR, with its whole-IDR GIN, the order of its blocks
and the sequence colored by residue type. *Min inter-cluster distance* is the
distance to the second-nearest centroid minus the distance to the nearest: large
means the IDR clearly has one grammar, close to 0 means two grammars fit about
equally well. Whole-IDR values come from step 3's full-sequence run, or from the
human IDRome reference when that mode was not run.

**Published** and **Multi-block**: one row per block. Each block is represented
by its *best window*: the 70 aa window centred inside the block that is closest
to the block's own GIN centroid. Its min inter-cluster distance is the block's
confidence. Positions are 0-based offsets inside the IDR
(`start`, `end` exclusive) and, when the IDR start is known, positions in the
protein. `best distance` is the squared distance of the best window in the block
to its centroid; lower means a closer match. The GIN cell uses the same color as
the protein diagram in step 6.

**Full sequence**: the three nearest centroids per IDR, with the squared distance
in brackets, so you can see whether the assignment is clear-cut or a close call.

Patterning z-scores come from random sequence scrambles (100 by default), so a
re-run can move a block boundary by a window or flip a close call. Raise
`NUM_SCRAMBLES` for steadier results.

In [ ]:
#@title Show the result tables { display-mode: "form" }
SEQUENCE_COLORS = "Kiersten"  #@param ["Kiersten", "Matt"]
SHOW_GIN_DEFINITIONS = True  #@param {type:"boolean"}

overview_blocks = next((results[k]["blocks"] for k in ("published", "multiblock")
                        if k in results and not results[k]["blocks"].empty), pd.DataFrame())
display(HTML("<h3 style='font-family: Arial, sans-serif'>Overview</h3>"))
display(nc.overview_table(inputs, idr_z, overview_blocks, SEQUENCE_COLORS))

for key, heading in (("published", "Published blocks"), ("multiblock", "Multi-block, recomputed")):
    if key in results:
        blocks = results[key]["blocks"]
        display(HTML(f"<h3 style='font-family: Arial, sans-serif'>{heading}</h3>"))
        display(nc.display_blocks(blocks, block_z.get(key)))
        if key == "multiblock":
            without = sorted(set(inputs["IDR_ID"]) - set(blocks.get("IDR_ID", [])))
            if without:
                print(f"No block passed the filters for: {', '.join(without)}")

if "fullseq" in results:
    display(HTML("<h3 style='font-family: Arial, sans-serif'>Full sequence</h3>"))
    display(nc.display_nearest_gins(results["fullseq"]["distances"]))

if SHOW_GIN_DEFINITIONS:
    display(HTML("<h3 style='font-family: Arial, sans-serif'>GIN definitions</h3>"))
    display(nc.gin_definitions_table())

## 5 · Grammar features

The 90 NARDINI+ z-scores (54 compositional, 36 patterning) as a heatmap, one row
per whole IDR or per block (its best window). Positive patterning z-scores mean
the two residue types are more segregated into blocks than in random scrambles
of the same sequence; negative ones mean they are more evenly mixed. z-scores are
capped at ±3, as in the GIN assignment. A feature is shown if at least one row
reaches the |z| cut-off.

In [ ]:
#@title Z-score heatmap { display-mode: "form" }
ROWS = "Blocks (best window)"  #@param ["Blocks (best window)", "Whole IDRs"]
BLOCKS_FROM = "Published"  #@param ["Published", "Multi-block"]
COLORMAP = "PRGn"  #@param ["PRGn", "RdBu_r", "PuOr", "Spectral_r"]
MAX_ABS_Z = 3  #@param {type:"number"}
MIN_ABS_Z = 1.0  #@param {type:"slider", min:0, max:3, step:0.1}
CLUSTERING = "Both"  #@param ["Both", "Rows only", "Features only", "Neither"]

def feature_rows(rows, blocks_from):
    if rows.startswith("Whole"):
        return idr_z, "whole_idrs"
    key = "published" if blocks_from == "Published" else "multiblock"
    z = block_z.get(key, pd.DataFrame())
    if z.empty and key == "published" and not block_z.get("multiblock", pd.DataFrame()).empty:
        print("No published blocks for these sequences; using the recomputed multi-block result.")
        key, z = "multiblock", block_z["multiblock"]
    return z, f"{key}_blocks"

zrows, tag = feature_rows(ROWS, BLOCKS_FROM)
if zrows.empty:
    print("Nothing to plot for this choice; run or enable the matching mode in step 3.")
else:
    fig = nc.zscore_heatmap(zrows, COLORMAP, MAX_ABS_Z, MIN_ABS_Z, CLUSTERING)
    if fig is not None:
        nc.save_figure(fig, OUT / "grammar_features" / f"{OUT.name}.{tag}.zscore_heatmap", FORMATS)
        zrows[["label"] + nc.FEATURES].to_csv(OUT / "grammar_features" / f"{OUT.name}.{tag}.zscores.csv", index=False)
        display(fig)
        plt.close(fig)

## 6 · Distance to every GIN

How far each IDR or block sits from all 30 centroids, nearest first. A steep
rise after the first point means one grammar fits much better than the rest; a
flat start means several are close.

In [ ]:
#@title Distance to the 30 centroids { display-mode: "form" }
ROWS = "Blocks (best window)"  #@param ["Blocks (best window)", "Whole IDRs"]
BLOCKS_FROM = "Published"  #@param ["Published", "Multi-block"]
PANELS_PER_ROW = 3  #@param {type:"integer"}

zrows, tag = feature_rows(ROWS, BLOCKS_FROM)
if zrows.empty:
    print("Nothing to plot for this choice; run or enable the matching mode in step 3.")
else:
    fig = nc.centroid_distance_figure(zrows, PANELS_PER_ROW)
    nc.save_figure(fig, OUT / "grammar_features" / f"{OUT.name}.{tag}.centroid_distances", FORMATS)
    display(fig)
    plt.close(fig)

## 7 · Colored sequence

Each residue is colored by chemical type (`analysis/plotting/plot_colored_sequence.py`).
Choose an IDR, then what to set in bold: a block, or the residues behind the most
non-random feature of the whole IDR or of a block's best window (for example
every K and R when pos-pos has the highest z-score, or the residues inside
E patches). Figures go to `colored_sequence/` in the results folder as PNG, PDF
and HTML.

In [ ]:
#@title Colored sequence viewer { display-mode: "form" }
COLOR_SET = "Kiersten"  #@param ["Kiersten", "Matt"]
WRAP_WIDTH = 60  #@param {type:"slider", min:30, max:100, step:10}
BLOCKS_FROM = "Published"  #@param ["Published", "Multi-block", "Full sequence"]
SAVE_EVERY_IDR = False  #@param {type:"boolean"}

import ipywidgets as widgets

key = {"Published": "published", "Multi-block": "multiblock", "Full sequence": "fullseq"}[BLOCKS_FROM]
if key == "published" and results.get("published", {}).get("blocks", pd.DataFrame()).empty and "multiblock" in results:
    print("No published blocks for these sequences; using the recomputed multi-block result.")
    key = "multiblock"
view_blocks = results.get(key, {}).get("blocks", pd.DataFrame())
view_block_z = block_z.get(key, pd.DataFrame())
seq_dir = OUT / "colored_sequence"


def draw_sequence(idr_id, highlight, save=True, show=True):
    record = inputs.set_index("IDR_ID").loc[idr_id].copy()
    record["IDR_ID"] = idr_id
    fig, html = nc.colored_sequence_figure(record, highlight, COLOR_SET, WRAP_WIDTH)
    if save:
        suffix = f".bold_{abs(hash(highlight)) % 10**6}" if highlight else ""
        prefix = seq_dir / f"{idr_id}{suffix}.colored_sequence"
        nc.save_figure(fig, prefix, FORMATS)
        Path(f"{prefix}.html").write_text(html)
    if show:
        display(fig)
    plt.close(fig)


if SAVE_EVERY_IDR:
    for idr in inputs["IDR_ID"]:
        draw_sequence(idr, "", show=False)
    print(f"Saved {len(inputs)} figure(s) to {seq_dir}")

pick_idr = widgets.Dropdown(options=list(inputs["IDR_ID"]), description="IDR",
                            layout=widgets.Layout(width="420px"))
pick_block = widgets.Dropdown(description="Bold", layout=widgets.Layout(width="560px"))
canvas = widgets.Output()


def on_idr(_=None):
    record = inputs.set_index("IDR_ID").loc[pick_idr.value].copy()
    record["IDR_ID"] = pick_idr.value
    pick_block.options = (nc.block_choices(view_blocks, pick_idr.value)
                          + nc.top_feature_choices(record, idr_z, view_block_z))
    pick_block.value = ""
    redraw()


def redraw(_=None):
    with canvas:
        clear_output(wait=True)
        draw_sequence(pick_idr.value, pick_block.value or "")


pick_idr.observe(on_idr, names="value")
pick_block.observe(redraw, names="value")
display(widgets.VBox([widgets.HBox([pick_idr, pick_block]), canvas]))
on_idr()

## 8 · Protein architecture

Each protein drawn as a grey bar with its UniProt domains (yellow), the IDR
(blue bar above) and the GIN blocks colored and numbered by GIN
(`analysis/plotting/plot_proteins.py`). Domains come from the bundled human
UniProt export, or from the UniProt API for other proteins. Sequences without a
UniProt ID and IDR start are drawn on their own. If an IDR runs past the UniProt
sequence length it was cut from another isoform, so that protein's domains are
left out and a note is printed.

In [ ]:
#@title Draw protein diagrams { display-mode: "form" }
PROTEINS = ""  #@param {type:"string"}
BLOCKS_FROM = "Published"  #@param ["Published", "Multi-block", "Full sequence"]
FETCH_DOMAINS_FROM_UNIPROT = True  #@param {type:"boolean"}
SHOW_DOMAINS = True  #@param {type:"boolean"}
FIGURE_WIDTH = 12  #@param {type:"slider", min:8, max:20, step:1}
MAX_TRACKS = 12  #@param {type:"integer"}
#@markdown Leave `PROTEINS` empty to draw every protein in this run, or list UniProt IDs (or IDR_IDs for sequences without one), comma-separated.

key = {"Published": "published", "Multi-block": "multiblock", "Full sequence": "fullseq"}[BLOCKS_FROM]
if key == "published" and results.get("published", {}).get("blocks", pd.DataFrame()).empty and "multiblock" in results:
    print("No published blocks for these sequences; using the recomputed multi-block result.")
    key = "multiblock"
plot_blocks = results.get(key, {}).get("blocks", pd.DataFrame())
if plot_blocks.empty:
    print(f"No {BLOCKS_FROM.lower()} blocks to draw; run or enable that mode in step 3.")
else:
    domains = nc.domain_table(plot_blocks, fetch_from_uniprot=FETCH_DOMAINS_FROM_UNIPROT)
    tracks = [p.strip() for p in PROTEINS.replace(";", ",").split(",") if p.strip()] or nc.track_ids(plot_blocks, domains)
    if len(tracks) > MAX_TRACKS:
        print(f"Drawing the first {MAX_TRACKS} of {len(tracks)} proteins; raise MAX_TRACKS for more.")
        tracks = tracks[:MAX_TRACKS]
    fig = nc.protein_figure(domains, tracks, show_domains=SHOW_DOMAINS, width=FIGURE_WIDTH)
    nc.save_figure(fig, OUT / "protein_plots" / f"{OUT.name}.{key}.protein_domains", FORMATS)
    domains.to_csv(OUT / "protein_plots" / f"{OUT.name}.{key}.domain_architecture.csv", index=False)
    display(fig)
    plt.close(fig)

## 9 · Exceptional grammars

Features where an IDR, or a block's best window, ranks among the most extreme in
the human IDRome: the top `RANK_CUTOFF` of 10,371 whole IDRs, or of the 10,592
published blocks. *High* ranks the largest z-scores first (blocky patterning,
enriched composition). *Low* ranks the smallest first, and only for features where
a low value carries meaning: patterning (well mixed) and R/K ratio, E/D ratio,
NCPR, hydrophobicity and isoelectric point. The IDR itself is left out of the
reference. Because z-scores are capped at ±3, a value at the cap ranks behind
every other IDR at the cap.

In [ ]:
#@title Rank against the human IDRome { display-mode: "form" }
ROWS = "Blocks (best window)"  #@param ["Blocks (best window)", "Whole IDRs"]
BLOCKS_FROM = "Published"  #@param ["Published", "Multi-block"]
RANK_CUTOFF = 150  #@param {type:"integer"}

zrows, tag = feature_rows(ROWS, BLOCKS_FROM)
if zrows.empty:
    print("Nothing to rank for this choice; run or enable the matching mode in step 3.")
else:
    level = "whole" if ROWS.startswith("Whole") else "blocks"
    ranks = nc.exceptional_features(zrows, level, RANK_CUTOFF)
    ranks.to_csv(OUT / "grammar_features" / f"{OUT.name}.{tag}.exceptional_features.csv", index=False)
    if ranks.empty:
        print(f"No feature ranks in the top {RANK_CUTOFF}.")
    else:
        display(ranks.style.hide(axis="index").format({"z": "{:.2f}"})
                .background_gradient(cmap="Blues_r", subset=["rank"]))

## 10 · Your IDRs against the rest of the IDRome

For each feature, a two-sample Kolmogorov–Smirnov test of your IDRs (or blocks)
against all other human IDRs (or published blocks). The heatmap shows the signed
−log10(p) for features with p < 0.05: positive when your set has the higher mean.
Use at least 50 IDRs; with a handful the test has little power. Set the |mean z|
filter to 1 to keep only features that are also non-random in your set.

In [ ]:
#@title Compare with the human IDRome { display-mode: "form" }
ROWS = "Whole IDRs"  #@param ["Whole IDRs", "Blocks (best window)"]
BLOCKS_FROM = "Published"  #@param ["Published", "Multi-block"]
MIN_ABS_MEAN_Z = 0.0  #@param {type:"slider", min:0, max:3, step:0.1}
COLORMAP = "PRGn"  #@param ["PRGn", "RdBu_r", "PuOr", "Spectral_r"]
MAX_SIGNED_LOG10_P = 10  #@param {type:"number"}

zrows, tag = feature_rows(ROWS, BLOCKS_FROM)
if zrows.empty:
    print("Nothing to compare for this choice; run or enable the matching mode in step 3.")
else:
    if len(zrows) < 50:
        print(f"Only {len(zrows)} rows; treat the result as indicative (50 or more recommended).")
    level = "whole" if ROWS.startswith("Whole") else "blocks"
    table = nc.compare_to_idrome(zrows, level, MIN_ABS_MEAN_Z)
    table.to_csv(OUT / "grammar_features" / f"{OUT.name}.{tag}.vs_idrome.csv", index=False)
    fig = nc.comparison_figure(table, COLORMAP, MAX_SIGNED_LOG10_P,
                               f"Features that differ from the rest of the human IDRome ({len(zrows)} {'blocks' if level == 'blocks' else 'IDRs'})")
    if fig is not None:
        nc.save_figure(fig, OUT / "grammar_features" / f"{OUT.name}.{tag}.vs_idrome", FORMATS)
        display(fig)
        plt.close(fig)

## 11 · Save the results

Everything above was written to `/content/nardini_results/<RUN_NAME>/`: the
pipeline CSVs (`*.out_windows.csv`, `*.out_summary.csv`, `*.out_dist_matrix.csv`),
the figures and the domain table. Colab deletes these files when the runtime
stops, so download the zip or copy it to Drive.

In [ ]:
#@title Download or copy to Drive { display-mode: "form" }
COPY_TO_DRIVE_FOLDER = ""  #@param {type:"string"}
#@markdown Leave empty to download the zip; otherwise give a folder in My Drive, e.g. `NARDINI_results`.

archive = nc.zip_results(OUT)
if COPY_TO_DRIVE_FOLDER.strip():
    from google.colab import drive
    drive.mount("/content/drive")
    target = Path("/content/drive/MyDrive") / COPY_TO_DRIVE_FOLDER.strip()
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy(archive, target / archive.name)
    print(f"Copied to {target / archive.name}")
elif IN_COLAB:
    from google.colab import files
    files.download(str(archive))
else:
    print(f"Saved {archive}")